# Notebook 1 — Transformasi Data BMKG Asli menjadi Dataset BMKG 2020–2025
### (Di Luar Pipeline CRISP-DM Utama)

Notebook ini **hanya** berisi transformasi: dari data mentah BMKG (Juli 2024 – Jan 2026, 549 baris) menjadi dataset lengkap rentang **2020–2025**. Fase CRISP-DM (Data Understanding, Data Preparation, Modeling, dst.) ada sepenuhnya di notebook terpisah `02_crisp_dm_prediksi_curah_hujan.ipynb`.

**Tentang label sumber data:** karena data BMKG resmi baru tersedia mulai Juli 2024, periode 2020 s/d Juli 2024 diisi lewat **augmentasi statistik** (bukan observasi lapangan) berbasis pola musiman dari data BMKG yang ada. Supaya tetap jujur secara metodologis, dataset akhir memberi label:
- `BMKG_OBSERVASI` — baris hasil pengukuran langsung BMKG
- `BMKG_AUGMENTASI` — baris hasil augmentasi statistik (disintesis dari pola BMKG_OBSERVASI, bukan pengukuran asli)

Keduanya tetap "bersumber dari BMKG" (augmentasi dibangun dari distribusi data BMKG asli, bukan dikarang bebas), tapi label ini penting dipertahankan supaya bisa dibedakan mana observasi langsung dan mana estimasi — ini wajib disebutkan di metodologi skripsi.

**Catatan integritas:** file mentah `laporan_iklim_harian_dataset.csv` hanya **dibaca**, tidak pernah ditulis ulang di sini.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Muat Data Mentah BMKG (Read-Only)

In [ ]:
RAW_PATH = 'laporan_iklim_harian_dataset.csv'

df_raw = pd.read_csv(RAW_PATH)
df_raw['TANGGAL'] = pd.to_datetime(df_raw['TANGGAL'])

print('Jumlah baris:', len(df_raw))
print('Rentang tanggal:', df_raw['TANGGAL'].min().date(), 'sampai', df_raw['TANGGAL'].max().date())
df_raw.head()


## 3. Penanganan Kode Error BMKG (Prasyarat Transformasi)

Kode `8888`/`9999` BMKG bukan nilai ukur asli (kode error alat/data tidak terukur), jadi harus dibersihkan dulu sebelum dipakai sebagai basis augmentasi statistik — kalau tidak, kode error ini akan ikut ter-sampling dan merusak hasil augmentasi.

In [ ]:
df_clean = df_raw.copy()
MISSING_CODES = [8888, 9999]
numeric_cols = ['TN', 'TX', 'TAVG', 'RH_AVG', 'RR', 'SS', 'FF_X', 'FF_AVG']

for col in numeric_cols:
    n_bad = df_clean[col].isin(MISSING_CODES).sum()
    if n_bad > 0:
        print(f'{col}: {n_bad} nilai kode BMKG diubah jadi NaN')
    df_clean[col] = df_clean[col].replace(MISSING_CODES, np.nan)

df_clean = df_clean.sort_values('TANGGAL').reset_index(drop=True).set_index('TANGGAL')
for col in numeric_cols:
    df_clean[col] = df_clean[col].interpolate(method='time', limit_direction='both')
df_clean = df_clean.reset_index()

df_clean['FF_X'] = df_clean['FF_X'].round().astype(int)
df_clean['FF_AVG'] = df_clean['FF_AVG'].round().astype(int)
df_clean['SUMBER'] = 'BMKG_OBSERVASI'

print('Sisa missing value:', df_clean[numeric_cols].isna().sum().sum())


## 4. Pola Musiman Bulanan (Basis Augmentasi)

In [ ]:
df_clean['bulan'] = df_clean['TANGGAL'].dt.month
df_clean['tahun'] = df_clean['TANGGAL'].dt.year

monthly_stats = df_clean.groupby('bulan')['RR'].agg(['mean', 'std', 'count'])
print(monthly_stats)


## 5. Augmentasi Data 2020-01-01 s/d 2024-07-02

**Metode: Seasonal Block Bootstrap + Jitter**
1. Untuk tiap tanggal kosong, ambil kumpulan baris `BMKG_OBSERVASI` di **bulan kalender yang sama**.
2. Pilih satu baris (hari utuh) secara acak — *block bootstrap* menjaga korelasi antar variabel (TN-TX-RH-RR) tetap realistis.
3. Tambahkan noise Gaussian kecil supaya tidak duplikat persis dari hari yang di-sampel.
4. Clip ke rentang fisik yang masuk akal, lalu bulatkan sesuai presisi asli.

In [ ]:
def buat_data_augmentasi(df_observasi, tanggal_mulai, tanggal_akhir, noise_scale=0.08, random_state=42):
    rng = np.random.default_rng(random_state)
    tanggal_range = pd.date_range(tanggal_mulai, tanggal_akhir, freq='D')
    pool_per_bulan = {b: df_observasi[df_observasi['bulan'] == b].reset_index(drop=True) for b in range(1, 13)}

    baris_augmentasi = []
    for tgl in tanggal_range:
        pool = pool_per_bulan[tgl.month]
        sample = pool.sample(n=1, random_state=rng.integers(0, 1_000_000)).iloc[0].copy()
        new_row = {'TANGGAL': tgl}
        for col in numeric_cols:
            std_col = df_observasi[col].std()
            jitter = rng.normal(0, noise_scale * std_col)
            new_row[col] = sample[col] + jitter
        baris_augmentasi.append(new_row)

    df_aug = pd.DataFrame(baris_augmentasi)
    df_aug['RR'] = df_aug['RR'].clip(lower=0).round(1)
    df_aug['RH_AVG'] = df_aug['RH_AVG'].clip(lower=0, upper=100).round(1)
    df_aug['SS'] = df_aug['SS'].clip(lower=0, upper=8).round(1)
    df_aug['TN'] = df_aug['TN'].round(1)
    df_aug['TX'] = df_aug['TX'].round(1)
    df_aug['TAVG'] = df_aug['TAVG'].round(1)
    df_aug['FF_X'] = df_aug['FF_X'].clip(lower=0).round().astype(int)
    df_aug['FF_AVG'] = df_aug['FF_AVG'].clip(lower=0).round().astype(int)
    df_aug['bulan'] = df_aug['TANGGAL'].dt.month
    df_aug['tahun'] = df_aug['TANGGAL'].dt.year
    df_aug['SUMBER'] = 'BMKG_AUGMENTASI'
    return df_aug


tanggal_mulai_aug = '2020-01-01'
tanggal_akhir_aug = df_clean['TANGGAL'].min() - pd.Timedelta(days=1)

df_augmentasi = buat_data_augmentasi(df_clean, tanggal_mulai_aug, tanggal_akhir_aug)
print('Jumlah baris augmentasi:', len(df_augmentasi))
df_augmentasi.head()


## 6. Gabungkan & Simpan Dataset BMKG 2020–2025

In [ ]:
kolom_final = ['TANGGAL', 'TN', 'TX', 'TAVG', 'RH_AVG', 'RR', 'SS', 'FF_X', 'FF_AVG', 'SUMBER']

df_final = pd.concat([df_augmentasi[kolom_final], df_clean[kolom_final]], ignore_index=True)
df_final = df_final.sort_values('TANGGAL').reset_index(drop=True)

print('Total baris:', len(df_final))
print('Rentang tanggal:', df_final['TANGGAL'].min().date(), '-', df_final['TANGGAL'].max().date())
print(df_final['SUMBER'].value_counts())


In [ ]:
OUTPUT_PATH = 'dataset_bmkg_2020_2025.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'Dataset disimpan ke: {OUTPUT_PATH}')
print('File ini menjadi input notebook ke-2 (CRISP-DM: Data Understanding s/d Deployment).')
df_final.head()


## Ringkasan

- `BMKG_OBSERVASI`: 549 baris (Jul 2024 – Jan 2026), pengukuran langsung BMKG (dibersihkan dari kode error).
- `BMKG_AUGMENTASI`: 1645 baris (Jan 2020 – Jul 2024), hasil augmentasi statistik dari pola bulanan `BMKG_OBSERVASI`.
- Kolom `SUMBER` **wajib dipertahankan** di dataset akhir dan **wajib disebutkan di metodologi skripsi** — ini standar praktik ilmiah saat memakai data augmentasi, bukan sesuatu yang perlu disembunyikan.
- **Lanjutkan ke `02_crisp_dm_prediksi_curah_hujan.ipynb`** dengan file `dataset_bmkg_2020_2025.csv` yang baru disimpan.